# MoCap Analysis — Transform to TFrame Coordinate System

**Workflow:**
- `multiple_rigid_body.py` is loaded once via `importlib`.
- It runs its `__main__` block which produces `rb_dfs` and `st_time`.
- This notebook uses those DataFrames directly — no CSV re-read.

**Pipeline:**
1. Imports
2. Load script → get `rb_dfs` and `st_time`
3. Verify DataFrames are ready
4. Inspect raw data
5. Define and run coordinate frame transform
6. Sanity checks
7. Print noark m5 position in tframe (metres)
8. Print table m1–m5 positions in tframe (metres)
9. 2D trajectory plot — NOARK marker 5 in table frame


## Cell 1 — Imports

In [1]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

print('Libraries loaded OK')

Libraries loaded OK


## Cell 2 — Load `multiple_rigid_body.py`

Walks up from the notebook directory to find the repo root,
then loads the script via `importlib`. The script's `__main__` block
runs automatically and populates `rb_dfs` and `st_time` into this
notebook's namespace.

In [2]:
import sys, importlib.util
from pathlib import Path

# ── locate this notebook's directory ─────────────────────────────────────────
try:
    _nb_file = Path(globals()['__vsc_ipynb_file__']).resolve()
    _nb_dir  = _nb_file.parent
except KeyError:
    _nb_dir = Path.cwd()

# ── walk up to find repo root ─────────────────────────────────────────────────
repo_root = _nb_dir
for candidate in [_nb_dir, *_nb_dir.parents]:
    if (candidate / 'vaideesh').exists() or (candidate / '.venv').exists():
        repo_root = candidate
        break

print(f'Repo root : {repo_root}')

# ── add script directory to sys.path so local imports inside the module work ──
_script_path = repo_root / 'vaideesh' / 'Analysis' / 'multiple_rigid_body.py'
assert _script_path.exists(), f'Script not found: {_script_path}'

_script_dir = str(_script_path.parent)
if _script_dir not in sys.path:
    sys.path.insert(0, _script_dir)

# ── load the module ───────────────────────────────────────────────────────────
spec = importlib.util.spec_from_file_location('multiple_rigid_body', _script_path)
mrb  = importlib.util.module_from_spec(spec)
sys.modules['multiple_rigid_body'] = mrb
spec.loader.exec_module(mrb)

# ── call the read function directly (since __main__ block doesn't run) ────────
FILE = "E:/Ragav/MS Bio Engineering/NOARK_backbone/mocap_data/table_frame.csv"

rb_dfs, st_time = mrb.read_3_rigid_body_csv(
    FILE,
    rb_names=["tframe", "noark", "table"],
)
rb_dfs = mrb.add_datetime_col_3rb(rb_dfs, st_time)
rb_dfs["tframe"], rb_dfs["noark"], rb_dfs["table"] = mrb.trunkate_3_dfs(
    rb_dfs["tframe"], rb_dfs["noark"], rb_dfs["table"], display_print=True
)

# ── bring helper functions into notebook namespace directly from mrb ───────────
add_datetime_col_3rb   = mrb.add_datetime_col_3rb
trunkate_3_dfs         = mrb.trunkate_3_dfs
get_rb_pos_cols        = mrb.get_rb_pos_cols
get_rb_rot_cols        = mrb.get_rb_rot_cols
get_rb_marker_name_3rb = mrb.get_rb_marker_name_3rb

print(f'Script loaded      : {_script_path}')
print(f'Capture start time : {st_time}')
print('rb_dfs keys        :', list(rb_dfs.keys()))

Repo root : E:\Ragav\MS Bio Engineering\NOARK_backbone
df_2 starts earlier, trimmed from index 0
df_2 ends later, trimmed to index 135
df_2 starts earlier, trimmed from index 0
df_2 ends later, trimmed to index 135
df_2 starts earlier, trimmed from index 0
df_2 ends later, trimmed to index 135
Script loaded      : E:\Ragav\MS Bio Engineering\NOARK_backbone\vaideesh\Analysis\multiple_rigid_body.py
Capture start time : 2026-04-17 17:06:12.485000
rb_dfs keys        : ['tframe', 'noark', 'table']


## Cell 3 — Verify DataFrames

The `__main__` block in `multiple_rigid_body.py` already calls
`add_datetime_col_3rb()` and `trunkate_3_dfs()`, so the DataFrames
are ready to use as-is. This cell just confirms their shapes and
that a `time` column is present.

In [3]:
print('DataFrames from multiple_rigid_body.py:')
print(f'{"Body":<10}  {"Shape":<15}  {"Has time col"}')
print('-' * 40)
for name, df in rb_dfs.items():
    has_time = 'time' in df.columns
    print(f'{name:<10}  {str(df.shape):<15}  {has_time}')

print('\nNaN rows per body:')
for name, df in rb_dfs.items():
    nan_rows = df.isna().any(axis=1).sum()
    print(f'  {name}: {nan_rows} NaN rows out of {len(df)}')

DataFrames from multiple_rigid_body.py:
Body        Shape            Has time col
----------------------------------------
tframe      (136, 27)        True
noark       (136, 31)        True
table       (136, 31)        True

NaN rows per body:
  tframe: 1 NaN rows out of 136
  noark: 1 NaN rows out of 136
  table: 1 NaN rows out of 136


## Cell 4 — Inspect raw DataFrames

In [5]:
print('── tframe ──')
display(rb_dfs['tframe'].head(30))

── tframe ──


,frame,seconds,tframe_rot_x,tframe_rot_y,tframe_rot_z,tframe_rot_w,tframe_pos_x,tframe_pos_y,tframe_pos_z,tframe_pos_err,...,tframe_marker_m2_mq,tframe_marker_m3_x,tframe_marker_m3_y,tframe_marker_m3_z,tframe_marker_m3_mq,tframe_marker_m4_x,tframe_marker_m4_y,tframe_marker_m4_z,tframe_marker_m4_mq,time
0,0,0.00,-0.000946,0.033981,-0.002022,-0.999420,0.074669,0.718711,0.482556,0.000779,...,0.735216,0.162471,0.641259,0.489042,0.913915,0.141485,0.789250,0.461756,0.843000,2026-04-17 17:06:12.485
1,1,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-17 17:06:12.495
2,2,0.02,-0.000786,0.034058,-0.001966,-0.999418,0.074670,0.718692,0.482578,0.000778,...,0.735000,0.162461,0.641232,0.489102,0.908744,0.141497,0.789217,0.461766,0.842706,2026-04-17 17:06:12.505
3,3,0.03,-0.000807,0.034119,-0.002065,-0.999415,0.074664,0.718668,0.482573,0.000825,...,0.735000,0.162469,0.641225,0.489104,0.908744,0.141479,0.789206,0.461771,0.842706,2026-04-17 17:06:12.515
4,4,0.04,-0.000799,0.034072,-0.002026,-0.999417,0.074666,0.718678,0.482580,0.000807,...,0.725718,0.162466,0.641228,0.489104,0.910754,0.141485,0.789211,0.461771,0.836358,2026-04-17 17:06:12.525
5,5,0.05,-0.000818,0.034103,-0.001972,-0.999416,0.074672,0.718693,0.482573,0.000793,...,0.731243,0.162464,0.641234,0.489100,0.908156,0.141500,0.789220,0.461772,0.839960,2026-04-17 17:06:12.535
6,6,0.06,-0.000847,0.034083,-0.001953,-0.999417,0.074673,0.718703,0.482569,0.000782,...,0.735011,0.162462,0.641240,0.489088,0.905613,0.141503,0.789229,0.461769,0.841366,2026-04-17 17:06:12.545
7,7,0.07,-0.000939,0.034003,-0.002055,-0.999419,0.074687,0.718735,0.482560,0.000758,...,0.735011,0.162493,0.641289,0.489051,0.905613,0.141498,0.789279,0.461762,0.841366,2026-04-17 17:06:12.555
8,8,0.08,-0.000895,0.034179,-0.001982,-0.999413,0.074668,0.718697,0.482565,0.000788,...,0.734625,0.162461,0.641239,0.489093,0.913158,0.141498,0.789229,0.461785,0.843855,2026-04-17 17:06:12.565
9,9,0.09,-0.000987,0.033827,-0.002011,-0.999425,0.074697,0.718712,0.482556,0.000794,...,0.735047,0.162499,0.641259,0.489009,0.913083,0.141507,0.789252,0.461741,0.850197,2026-04-17 17:06:12.575


In [6]:
print('── noark ──')
display(rb_dfs['noark'].head(30))

── noark ──


,frame,seconds,noark_rot_x,noark_rot_y,noark_rot_z,noark_rot_w,noark_pos_x,noark_pos_y,noark_pos_z,noark_pos_err,...,noark_marker_m3_mq,noark_marker_m4_x,noark_marker_m4_y,noark_marker_m4_z,noark_marker_m4_mq,noark_marker_m5_x,noark_marker_m5_y,noark_marker_m5_z,noark_marker_m5_mq,time
0,0,0.00,0.026926,0.817206,-0.039622,-0.574352,0.552702,0.784215,0.488730,0.000673,...,0.930199,0.574371,0.772861,0.533571,0.736747,0.446474,0.839150,0.527338,0.837144,2026-04-17 17:06:12.485
1,1,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-17 17:06:12.495
2,2,0.02,0.026893,0.817219,-0.039707,-0.574329,0.552684,0.784215,0.488731,0.000683,...,0.929978,0.574353,0.772869,0.533574,0.740727,0.446447,0.839138,0.527329,0.832648,2026-04-17 17:06:12.505
3,3,0.03,0.026957,0.817191,-0.039627,-0.574371,0.552695,0.784230,0.488736,0.000683,...,0.929978,0.574365,0.772877,0.533577,0.740727,0.446472,0.839172,0.527347,0.832648,2026-04-17 17:06:12.515
4,4,0.04,0.026823,0.817234,-0.039666,-0.574314,0.552692,0.784212,0.488733,0.000678,...,0.927623,0.574360,0.772861,0.533576,0.736187,0.446449,0.839122,0.527335,0.833094,2026-04-17 17:06:12.525
5,5,0.05,0.026826,0.817216,-0.039804,-0.574330,0.552701,0.784237,0.488740,0.000669,...,0.934592,0.574373,0.772900,0.533584,0.737638,0.446452,0.839139,0.527335,0.840186,2026-04-17 17:06:12.535
6,6,0.06,0.026823,0.817236,-0.039660,-0.574311,0.552691,0.784232,0.488735,0.000676,...,0.930398,0.574358,0.772880,0.533577,0.737253,0.446448,0.839142,0.527336,0.836609,2026-04-17 17:06:12.545
7,7,0.07,0.026945,0.817215,-0.039526,-0.574345,0.552695,0.784231,0.488730,0.000676,...,0.930398,0.574361,0.772868,0.533570,0.737253,0.446475,0.839176,0.527342,0.836609,2026-04-17 17:06:12.555
8,8,0.08,0.026925,0.817216,-0.039671,-0.574334,0.552708,0.784236,0.488745,0.000687,...,0.928359,0.574376,0.772887,0.533588,0.735246,0.446477,0.839167,0.527345,0.835238,2026-04-17 17:06:12.565
9,9,0.09,0.026647,0.817223,-0.039061,-0.574379,0.552662,0.784339,0.488848,0.000729,...,0.861710,0.574330,0.772925,0.533674,0.745083,0.446446,0.839246,0.527526,0.884779,2026-04-17 17:06:12.575


In [8]:
print('── table ──')
display(rb_dfs['table'].head(30))

── table ──


,frame,seconds,table_rot_x,table_rot_y,table_rot_z,table_rot_w,table_pos_x,table_pos_y,table_pos_z,table_pos_err,...,table_marker_m3_mq,table_marker_m4_x,table_marker_m4_y,table_marker_m4_z,table_marker_m4_mq,table_marker_m5_x,table_marker_m5_y,table_marker_m5_z,table_marker_m5_mq,time
0,0,0.00,0.000710,0.002674,0.000164,-0.999996,0.186785,0.723478,0.904015,0.000970,...,0.431160,-0.321982,0.722480,1.214657,0.907083,0.664479,0.709349,0.473975,0.848946,2026-04-17 17:06:12.485
1,1,0.01,0.000584,0.002683,0.000171,-0.999996,0.186726,0.723366,0.904030,0.001117,...,0.443356,-0.322047,0.722296,1.214662,0.950930,0.664428,0.709338,0.473996,0.847106,2026-04-17 17:06:12.495
2,2,0.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-17 17:06:12.505
3,3,0.03,0.000744,0.002667,0.000137,-0.999996,0.186788,0.723490,0.904023,0.000968,...,0.443356,-0.321976,0.722485,1.214672,0.950930,0.664476,0.709357,0.473977,0.847106,2026-04-17 17:06:12.515
4,4,0.04,0.000629,0.002672,0.000173,-0.999996,0.186724,0.723373,0.904027,0.001118,...,0.431152,-0.322043,0.722334,1.214670,0.903163,0.664416,0.709304,0.473983,0.850733,2026-04-17 17:06:12.525
5,5,0.05,0.000715,0.002669,0.000148,-0.999996,0.186788,0.723496,0.904027,0.000975,...,0.430660,-0.321977,0.722484,1.214674,0.905098,0.664477,0.709377,0.473982,0.847499,2026-04-17 17:06:12.535
6,6,0.06,0.000719,0.002670,0.000166,-0.999996,0.186784,0.723487,0.904021,0.000971,...,0.430519,-0.321981,0.722496,1.214666,0.907416,0.664475,0.709348,0.473977,0.851750,2026-04-17 17:06:12.545
7,7,0.07,0.000741,0.002663,0.000166,-0.999996,0.186784,0.723486,0.904026,0.000970,...,0.430519,-0.321976,0.722509,1.214679,0.907416,0.664468,0.709328,0.473976,0.851750,2026-04-17 17:06:12.555
8,8,0.08,0.000706,0.002675,0.000159,-0.999996,0.186786,0.723497,0.904023,0.000973,...,0.433205,-0.321982,0.722491,1.214664,0.904540,0.664481,0.709375,0.473984,0.850281,2026-04-17 17:06:12.565
9,9,0.09,0.000704,0.002665,0.000169,-0.999996,0.186783,0.723492,0.904027,0.000971,...,0.434546,-0.321978,0.722495,1.214678,0.908310,0.664469,0.709363,0.473978,0.848389,2026-04-17 17:06:12.575


## Cell 5 — Define `transform_to_tframe()`

**Coordinate frame construction (per frame, from mocap marker positions):**

| Step | Detail |
|---|---|
| Origin | `tframe_marker_1` position in mocap frame |
| X axis | `tframe_marker_3 − origin`, normalised |
| Y axis | `tframe_marker_2 − origin`, Gram-Schmidt orthogonalised then normalised |
| Z axis | `X × Y` (right-handed) |
| Translation reference | `tframe_marker_3` → becomes **(0, 0, 0)** in table frame |

**Transform formula:** `p_table = R_mocap2table.T @ (p_mocap − t_mocap)`  
where `R_mocap2table` columns are `[vxnorm, vynorm, vznorm]` and `t_mocap = tvec(tframe_marker_3)`.

**Units:** Motive exports positions in **metres** — no conversion needed.  
**Column suffix `_m`** makes the unit explicit in every position column.


In [ ]:
def transform_to_tframe(rb_dfs):
    tframe_df = rb_dfs['tframe'].copy()
    noark_df  = rb_dfs['noark'].copy()
    table_df  = rb_dfs['table'].copy()

    # ── Per-frame: build table coordinate frame from tframe marker positions ──
    # marker positions are already in mocap (world) frame
    m1 = tframe_df[['tframe_marker_m1_x', 'tframe_marker_m1_y', 'tframe_marker_m1_z']].values.astype(float)
    m2 = tframe_df[['tframe_marker_m2_x', 'tframe_marker_m2_y', 'tframe_marker_m2_z']].values.astype(float)
    m3 = tframe_df[['tframe_marker_m3_x', 'tframe_marker_m3_y', 'tframe_marker_m3_z']].values.astype(float)

    # Origin of table frame = tframe_marker_1 (in mocap frame)
    org = m1                             # shape (N, 3)

    # X direction: marker_3 − origin
    v1 = m3 - org

    # Y direction: marker_2 − origin
    v2 = m2 - org

    # Gram-Schmidt orthonormalisation
    vxnorm  = v1 / np.linalg.norm(v1, axis=1, keepdims=True)          # X unit vector
    vycap   = v2 - np.sum(v2 * vxnorm, axis=1, keepdims=True) * vxnorm
    vynorm  = vycap / np.linalg.norm(vycap, axis=1, keepdims=True)    # Y unit vector
    vznorm  = np.cross(vxnorm, vynorm)                                 # Z = X × Y (right-handed)

    # R_mocap2table: shape (N, 3, 3)  — columns are [vx, vy, vz] in mocap frame
    # R.T = R^{-1} rotates FROM mocap TO table
    mocap_rot_mat = np.stack([vxnorm, vynorm, vznorm], axis=-1)        # (N, 3, 3)

    # Translation vector: tframe_marker_3 is the TABLE FRAME ORIGIN in mocap frame
    mocap_tvec_vec = m3                                                 # shape (N, 3)

    # ── Helper: transform a (N,3) array of mocap positions to table frame ────
    def pos_to_tframe(p_mocap):
        # p_table = R.T @ (p_mocap − t)
        # einsum: for each row i: R[i].T @ (p_mocap[i] - t[i])
        delta = p_mocap - mocap_tvec_vec                               # (N, 3)
        return np.einsum('nij,nj->ni', mocap_rot_mat.transpose(0, 2, 1), delta)  # (N, 3)

    # ── noark ─────────────────────────────────────────────────────────────────
    quat_cols_noark = ['noark_rot_x', 'noark_rot_y', 'noark_rot_z', 'noark_rot_w']
    quat_cols_table = ['table_rot_x', 'table_rot_y', 'table_rot_z', 'table_rot_w']

    noark_result = pd.DataFrame({
        'frame'  : noark_df['frame'].values,
        'seconds': noark_df['seconds'].values,
        'time'   : noark_df['time'].values,
    })

    p_m5    = noark_df[['noark_marker_m5_x',
                         'noark_marker_m5_y',
                         'noark_marker_m5_z']].values.astype(float)
    p_m5_tf = pos_to_tframe(p_m5)
    noark_result['noark_m5_tf_x_m'] = p_m5_tf[:, 0]
    noark_result['noark_m5_tf_y_m'] = p_m5_tf[:, 1]
    noark_result['noark_m5_tf_z_m'] = p_m5_tf[:, 2]

    # ── table ─────────────────────────────────────────────────────────────────
    table_result = pd.DataFrame({
        'frame'  : table_df['frame'].values,
        'seconds': table_df['seconds'].values,
        'time'   : table_df['time'].values,
    })

    for i in range(1, 6):
        p_world     = table_df[[f'table_marker_m{i}_x',
                                 f'table_marker_m{i}_y',
                                 f'table_marker_m{i}_z']].values.astype(float)
        p_tf_coords = pos_to_tframe(p_world)
        table_result[f'table_m{i}_tf_x_m'] = p_tf_coords[:, 0]
        table_result[f'table_m{i}_tf_y_m'] = p_tf_coords[:, 1]
        table_result[f'table_m{i}_tf_z_m'] = p_tf_coords[:, 2]

    return {
        'noark_in_tframe': noark_result.reset_index(drop=True),
        'table_in_tframe': table_result.reset_index(drop=True),
        # expose for sanity checks
        '_mocap_rot_mat' : mocap_rot_mat,
        '_mocap_tvec_vec': mocap_tvec_vec,
        '_tframe_m3_mocap': m3,
    }


## Cell 6 — Run the transformation

In [ ]:
tframe_dfs = transform_to_tframe(rb_dfs)

print('noark_in_tframe shape :', tframe_dfs['noark_in_tframe'].shape)
print('table_in_tframe shape :', tframe_dfs['table_in_tframe'].shape)
print('\nnoark_in_tframe columns:')
print(tframe_dfs['noark_in_tframe'].columns.tolist())
print('\ntable_in_tframe columns:')
print(tframe_dfs['table_in_tframe'].columns.tolist())

## Cell 7 — Inspect transformed DataFrames

In [ ]:
print('── noark in tframe ──')
display(tframe_dfs['noark_in_tframe'].head())

In [ ]:
print('── table in tframe ──')
display(tframe_dfs['table_in_tframe'].head())

## Cell 8 — Sanity checks

In [ ]:
# ── Sanity checks for marker-based table frame transform ─────────────────────

tframe_extras = tframe_dfs   # includes _mocap_rot_mat, _mocap_tvec_vec, _tframe_m3_mocap

mocap_rot_mat   = tframe_extras['_mocap_rot_mat']    # (N, 3, 3)
mocap_tvec_vec  = tframe_extras['_mocap_tvec_vec']   # (N, 3)  = tframe_marker_3
tframe_m3_mocap = tframe_extras['_tframe_m3_mocap']  # (N, 3)

# ── Check 1: tframe_marker_3 (origin) transforms to (0, 0, 0) ────────────────
delta_m3    = tframe_m3_mocap - mocap_tvec_vec        # should be all zeros
p_m3_tf     = np.einsum('nij,nj->ni',
                          mocap_rot_mat.transpose(0, 2, 1),
                          delta_m3)
max_pos_err = np.nanmax(np.abs(p_m3_tf))
print(f'Check 1 — tframe_marker_3 → (0,0,0) : max error = {max_pos_err:.2e} m  →  {"PASS ✓" if max_pos_err < 1e-10 else "FAIL ✗"}')

# ── Check 2: rotation matrix columns are unit vectors ────────────────────────
col_norms = np.linalg.norm(mocap_rot_mat, axis=1)     # (N, 3)  norms of each column
max_norm_err = np.nanmax(np.abs(col_norms - 1.0))
print(f'Check 2 — R columns are unit vectors : max deviation = {max_norm_err:.2e}  →  {"PASS ✓" if max_norm_err < 1e-6 else "FAIL ✗"}')

# ── Check 3: rotation matrix is orthogonal (R.T @ R ≈ I) ────────────────────
RtR       = np.einsum('nij,njk->nik',
                       mocap_rot_mat.transpose(0, 2, 1),
                       mocap_rot_mat)                  # (N, 3, 3)
identity  = np.eye(3)[None].repeat(len(RtR), axis=0)
max_orth_err = np.nanmax(np.abs(RtR - identity))
print(f'Check 3 — R.T @ R = I (orthogonality) : max deviation = {max_orth_err:.2e}  →  {"PASS ✓" if max_orth_err < 1e-6 else "FAIL ✗"}')

# ── Check 4: right-handedness det(R) ≈ +1 ───────────────────────────────────
dets    = np.linalg.det(mocap_rot_mat)
max_det_err = np.nanmax(np.abs(dets - 1.0))
print(f'Check 4 — det(R) = +1 (right-handed)  : max deviation = {max_det_err:.2e}  →  {"PASS ✓" if max_det_err < 1e-6 else "FAIL ✗"}')

print(f'\n(checked over {len(mocap_rot_mat)} frames)')


In [ ]:
# (Quaternion-norm check removed — new transform is purely positional/marker-based,
#  no quaternion output is produced. Add orientation output here if needed later.)
print('No quaternion output in this transform — skipping norm check.')


## Cell 9 — noark m5 position in tframe (metres)

In [ ]:
# Cell 9 — noark m5
df_n = tframe_dfs['noark_in_tframe']
print('noark Marker 5 — position in tframe coordinate system (metres)')
print(df_n[['frame','seconds','noark_m5_tf_x_m','noark_m5_tf_y_m','noark_m5_tf_z_m']].to_string(index=False))
print('\nSummary statistics (metres):')
display(df_n[['noark_m5_tf_x_m','noark_m5_tf_y_m','noark_m5_tf_z_m']].describe().round(6))

## Cell 10 — Table marker m1–m5 positions in tframe (metres)

In [ ]:
# Cell 10 — table m1–m5
df_t = tframe_dfs['table_in_tframe']
for i in range(1, 6):
    cols = [f'table_m{i}_tf_x_m', f'table_m{i}_tf_y_m', f'table_m{i}_tf_z_m']
    print(f'\ntable Marker {i} — position in tframe coordinate system (metres)')
    print(df_t[['frame','seconds'] + cols].to_string(index=False))
    print(f'\nSummary statistics — Marker {i} (metres):')
    display(df_t[cols].describe().round(6))

## Cell 11 — 2D Trajectory of NOARK Marker 5 in Table Frame

Plots X–Y position of `noark_marker_m5` in the table coordinate frame.  
- **Colour** encodes time progression (blue → early, red → late).  
- **▲ triangles** mark the four table-corner reference points (P1–P4) derived from `table_m1–m4` mean positions.  
- Axes are converted to **cm** for readability.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# ── pull data ─────────────────────────────────────────────────────────────────
df_n = tframe_dfs['noark_in_tframe'].copy()
df_t = tframe_dfs['table_in_tframe'].copy()

# convert m → cm
x_cm = df_n['noark_m5_tf_x_m'].values * 100.0
y_cm = df_n['noark_m5_tf_y_m'].values * 100.0

# ── colour map: time progression ─────────────────────────────────────────────
t_norm = (np.arange(len(x_cm)) / max(len(x_cm) - 1, 1))   # 0 → 1

# ── table corner reference points (mean over all frames, m → cm) ─────────────
corner_labels = ['P1', 'P2', 'P3', 'P4']
corner_xy = []
for i in range(1, 5):
    cx = df_t[f'table_m{i}_tf_x_m'].mean() * 100.0
    cy = df_t[f'table_m{i}_tf_y_m'].mean() * 100.0
    corner_xy.append((cx, cy))

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

sc = ax.scatter(
    x_cm, y_cm,
    c=t_norm,
    cmap='coolwarm',
    s=18,
    alpha=0.75,
    zorder=2,
    label='NOARK m5 (MoCap)',
)

# colourbar
cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label('Time progression  (0 = start, 1 = end)', fontsize=9)

# table corner markers
for (cx, cy), lbl in zip(corner_xy, corner_labels):
    ax.plot(cx, cy, marker='^', markersize=12, color='black', zorder=5)
    ax.annotate(
        lbl,
        xy=(cx, cy),
        xytext=(0, 8),
        textcoords='offset points',
        ha='center', va='bottom',
        fontsize=9, fontweight='bold',
    )

ax.set_xlabel('X (cm)', fontsize=11)
ax.set_ylabel('Y (cm)', fontsize=11)
ax.set_title('2D Trajectory of NOARK Marker 5 in Table Frame', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f'\nTrajectory extents:')
print(f'  X : {x_cm.min():.1f} → {x_cm.max():.1f} cm  (range {x_cm.max()-x_cm.min():.1f} cm)')
print(f'  Y : {y_cm.min():.1f} → {y_cm.max():.1f} cm  (range {y_cm.max()-y_cm.min():.1f} cm)')
print(f'  N frames plotted: {len(x_cm)}')
